### Import packages

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('Set2')

### Select sheet

In [ ]:
DATA_FILE = '/path/to/CRISPR/Data/TGEV-筛选结果分析-2023-12-5.xls'
SHEET_NAME = 'zlu_lib'  # available: 'zlu_lib', 'Rep1', 'Rep2', 'Rep3', 'plasmid', 'cell pool'

# list all available sheets
import pandas as pd
xl = pd.ExcelFile(DATA_FILE)
available_sheets = xl.sheet_names

print(f'\nAvailable Sheets ({len(available_sheets)}):')
for i, sheet in enumerate(available_sheets, 1):
    marker = ' ← Current' if sheet == SHEET_NAME else ''
    print(f'  {i}. {sheet}{marker}')

In [ ]:
# get data
df = pd.read_excel(DATA_FILE, sheet_name=SHEET_NAME)

for i, col in enumerate(df.columns, 1):
    print(f'  {i}. {col}')

# Structure check (after loading)
expected_cols_all = ['sgRNA','seq','GeneID_x','GeneID_y','PigGeCKO cells_read1','PigGeCKO cells_read2',
                     'PigGeCKO cells_read1_norm','PigGeCKO cells_read2_norm','average(PigGeCKO cells)',
                     'read1','read2','read1_norm','read2_norm','average','Fold_change','log2(Fold_change)']
df.head()

### calculate p-value
### t-test of normalized reads

In [ ]:
def calculate_pvalues(df):
    """Calculate t-test p-values between before and after screening."""
    def safe_ttest(row):
        control = [row['PigGeCKO cells_read1_norm'], row['PigGeCKO cells_read2_norm']]
        treatment = [row['read1_norm'], row['read2_norm']]
        try:
            return stats.ttest_ind(treatment, control, equal_var=False, nan_policy='omit').pvalue or 1.0
        except Exception:
            return 1.0

    return df.apply(safe_ttest, axis=1)


# === Main logic ===
print('Calculating p-values...')
df['p_value'] = calculate_pvalues(df)
df['-log10(p_value)'] = -np.log10(df['p_value'] + 1e-300)


In [ ]:
# ============== parameters ==============

# Fold Change（log2）
FC_THRESHOLD = 1

# p-value
P_THRESHOLD = 0.05 

# volcano plot colors
COLOR_ENRICHED = '#d62728'  # enriched(red)
COLOR_DEPLETED = '#1f77b4'  # depleted(blue)
COLOR_NS = '#7f7f7f'        # not significant(gray)

# label top N enriched and depleted genes
ANNOTATE_TOP_N = 10  # label top N enriched and depleted genes, set 0 to not label
ANNOTATE_DEPLETED = True  # label all depleted genes

# figure size
FIGURE_SIZE = (14, 10)

# figure language
USE_ENGLISH = True  # True=English, False=Chinese

# gene ID column name (auto detect)
if 'GeneID_y' in df.columns:
    GENE_COL = 'GeneID_y'  # Use GeneID_y for complete gene symbol mapping
elif 'gene' in df.columns:
    GENE_COL = 'gene'
else:
    GENE_COL = df.columns[2]  # default 3rd column

In [ ]:

labels = {
    'up': 'Enriched',
    'down': 'Depleted',
    'ns': 'Not Significant'
    }

# classify genes based on thresholds
df['significant'] = labels['ns']
df.loc[(df['log2(Fold_change)'] > FC_THRESHOLD) & (df['p_value'] < P_THRESHOLD), 'significant'] = labels['up']
df.loc[(df['log2(Fold_change)'] < -FC_THRESHOLD) & (df['p_value'] < P_THRESHOLD), 'significant'] = labels['down']

# Count each category
n_up = (df['significant'] == labels['up']).sum()
n_down = (df['significant'] == labels['down']).sum()
n_ns = (df['significant'] == labels['ns']).sum()

In [ ]:
# generate output file name
suffix = f'{SHEET_NAME}_FC{FC_THRESHOLD}_p{P_THRESHOLD}'
# save data
output_data = f'/path/to/CRISPR/results/CRISPR_results_{suffix}.csv'